# IFA2 — Bivariate OU Model (Methodology 1, v1)

**Reference:** Abadie & Chamorro (2021) bivariate mean-reverting jump-diffusion.

GB and FR electricity prices are modelled as two separate Ornstein-Uhlenbeck processes with correlated Brownian innovations. The spread is computed as the difference: S(t,h) = P_GB(t,h) − P_FR(t,h).

**Estimation:** Full available sample — Dec 2021 to Jul 2026 — used as-is. No window selection, no crisis filtering, no price-path assumptions. All parameters (level, seasonal, OU dynamics, jumps) are estimated from the data.

**Output:** Annual P10/P50/P90 revenues in 2016/17-RPI-real GBP millions, 2026–2045.

In [1]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi':120,'axes.spines.top':False,
                     'axes.spines.right':False,'axes.grid':True,
                     'grid.linewidth':0.3,'grid.alpha':0.5,'font.size':10})

IC_DIR      = Path('/Users/aadesh/Documents/IC')
EXCEL_PATH  = IC_DIR / 'GB-FR07.xlsx'
OUTPUT_DIR  = IC_DIR / 'M1'
OUTPUT_DIR.mkdir(exist_ok=True)

N_PATHS         = 10_000
N_PROJ_YEARS    = 20
PROJ_START_YEAR = 2026
RANDOM_SEED     = 42
BATCH           = 500

CAPACITY_MW   = 1_000
AVAILABILITY  = 0.9659
CAPTURE_RATIO = 0.259

# RPI deflation — FPA Input row 97 (CHAW annual average, Jan 1987=100)
_ALL_RPI = {
    2021:310.794, 2022:320.864, 2023:331.260, 2024:341.993,
    2025:353.073, 2026:364.513, 2027:376.323, 2028:388.516,
    2029:401.104, 2030:414.099, 2031:427.516, 2032:441.368,
    2033:455.668, 2034:470.432, 2035:485.674, 2036:501.410,
    2037:517.655, 2038:534.427, 2039:551.743, 2040:569.619,
    2041:588.075, 2042:607.129, 2043:626.799, 2044:647.108, 2045:668.074,
}
_RPI_1617 = 264.992

print(f'Config: {N_PATHS:,} paths  |  Projection {PROJ_START_YEAR}–{PROJ_START_YEAR+N_PROJ_YEARS-1}')

Config: 10,000 paths  |  Projection 2026–2045


---
## Section 1: Data Loading

Source: `GB-FR07.xlsx` — Bloomberg hourly GB (GBP/MWh) and FR (EUR/MWh) prices + EUR/GBP FX. All prices deflated to 2016/17-RPI-real using FPA row 97 (same basis as M2).

In [2]:
def _wide_to_hourly(sheet, vcol):
    df = xl.parse(sheet).rename(columns={'Unnamed: 0':'date'})
    df['date'] = pd.to_datetime(df['date'])
    hc = sorted([c for c in df.columns if str(c).startswith('H') and str(c)[1:].isdigit()],
                key=lambda x: int(x[1:]))
    m  = df.melt(id_vars='date', value_vars=hc, var_name='hcol', value_name=vcol)
    m['ts'] = m['date'] + pd.to_timedelta(m['hcol'].str[1:].astype(int)-1, unit='h')
    return m.set_index('ts')[vcol].sort_index().astype(float)

xl   = pd.ExcelFile(EXCEL_PATH)
gb_h = _wide_to_hourly('GB_FINAL', 'gb_price_gbp')
fr_h = _wide_to_hourly('FR_FINAL', 'fr_price_eur')

fx_raw = xl.parse('GBP-EUR'); fx_raw.columns = ['date','fx_eur_gbp']
fx_raw['date'] = pd.to_datetime(fx_raw['date'])
fx_d = (fx_raw.dropna().set_index('date')['fx_eur_gbp']
        .reindex(pd.date_range(fx_raw['date'].min(), fx_raw['date'].max(), freq='D'))
        .ffill())

panel = pd.concat([gb_h, fr_h], axis=1).dropna()
panel = panel[~panel.index.duplicated(keep='last')]
panel = panel.reindex(pd.date_range(panel.index.min(), panel.index.max(), freq='h'))
panel = panel.interpolate(method='linear', limit=4).dropna()
panel['fx_eur_gbp']   = fx_d.reindex(panel.index.normalize()).values
panel['fx_eur_gbp']   = panel['fx_eur_gbp'].ffill().bfill()
panel['fr_price_gbp'] = panel['fr_price_eur'] * panel['fx_eur_gbp']
panel['spread_gbp']   = panel['gb_price_gbp'] - panel['fr_price_gbp']

# RPI deflation to 2016/17-real
_defl = np.array([_RPI_1617 / _ALL_RPI.get(d.year, _ALL_RPI[max(_ALL_RPI)])
                  for d in panel.index])
panel['gb_price_gbp'] *= _defl
panel['fr_price_gbp'] *= _defl
panel['spread_gbp']   *= _defl
panel = panel.dropna(subset=['gb_price_gbp','fr_price_gbp','spread_gbp'])

daily = panel[['gb_price_gbp','fr_price_gbp','spread_gbp']].resample('D').mean().dropna()
n_days = len(daily)
print(f'Sample: {daily.index[0].date()} → {daily.index[-1].date()}  ({n_days} days)')
print(f'Prices in 2016/17-RPI-real GBP/MWh')
print()
print(f'  {"Series":<22}  {"Mean":>8}  {"Std":>8}  {"Min":>8}  {"Max":>8}')
print('  ' + '-'*55)
for col, lbl in [('gb_price_gbp','GB'),('fr_price_gbp','FR'),('spread_gbp','Spread GB-FR')]:
    s = daily[col]
    print(f'  {lbl:<22}  {s.mean():>8.2f}  {s.std():>8.2f}  {s.min():>8.2f}  {s.max():>8.2f}')

Sample: 2021-12-09 → 2026-07-07  (1672 days)
Prices in 2016/17-RPI-real GBP/MWh

  Series                      Mean       Std       Min       Max
  -------------------------------------------------------
  GB                        104.35     73.03      0.36    558.24
  FR                         80.97     79.76    -25.95    524.64
  Spread GB-FR               23.39     29.28   -198.49    218.35


---
## Section 2: Long-Run Mean (Theta)

Theta is the unconditional mean of each price series estimated from the **full sample**. This is the level to which prices mean-revert in the OU process. No adjustment or recalibration is applied.

In [3]:
theta_gb = daily['gb_price_gbp'].mean()
theta_fr = daily['fr_price_gbp'].mean()

print('Long-run mean (2016/17-RPI-real GBP/MWh, full sample):')
print(f'  theta_GB     = {theta_gb:.2f}')
print(f'  theta_FR     = {theta_fr:.2f}')
print(f'  theta_spread = {theta_gb - theta_fr:.2f}')

Long-run mean (2016/17-RPI-real GBP/MWh, full sample):
  theta_GB     = 104.35
  theta_FR     = 80.97
  theta_spread = 23.39


---
## Section 3: Seasonal Decomposition

Annual and semi-annual Fourier harmonics (4 terms each) fitted **separately** for GB and FR on the full-sample demeaned daily prices. The seasonal component f_S_r(t) is subtracted before OU estimation and added back in projection.

In [4]:
def fit_fourier(daily_col, theta):
    s   = daily_col - theta
    doy = daily_col.index.dayofyear.values
    t   = 2 * np.pi * doy / 365.25
    X   = np.column_stack([np.sin(t), np.cos(t), np.sin(2*t), np.cos(2*t)])
    c,  *_ = np.linalg.lstsq(X, s.values, rcond=None)
    def predict(idx):
        t2 = 2 * np.pi * idx.dayofyear / 365.25
        return c[0]*np.sin(t2) + c[1]*np.cos(t2) + c[2]*np.sin(2*t2) + c[3]*np.cos(2*t2)
    fitted = pd.Series(X @ c, index=daily_col.index)
    return fitted, c, predict

f_gb, c_gb, fn_gb = fit_fourier(daily['gb_price_gbp'], theta_gb)
f_fr, c_fr, fn_fr = fit_fourier(daily['fr_price_gbp'], theta_fr)

z_gb = daily['gb_price_gbp'] - theta_gb - f_gb
z_fr = daily['fr_price_gbp'] - theta_fr - f_fr

print('Fourier seasonal (full sample):')
print(f'  GB [sin1,cos1,sin2,cos2]: {c_gb.round(3)}')
print(f'  FR [sin1,cos1,sin2,cos2]: {c_fr.round(3)}')
print(f'  Residual std post-seasonal:  GB={z_gb.std():.2f}  FR={z_fr.std():.2f} GBP/MWh')

Fourier seasonal (full sample):
  GB [sin1,cos1,sin2,cos2]: [-7.932  5.021 14.716  4.894]
  FR [sin1,cos1,sin2,cos2]: [-15.292   1.401  17.633   6.024]
  Residual std post-seasonal:  GB=71.85  FR=77.89 GBP/MWh


---
## Section 4: Jump Detection & OU Estimation

Residuals exceeding ±3σ on either series are classified as jump dates. Jump events are separated out for the Poisson/exponential jump model; the remaining clean residuals are used for AR(1) OU estimation. All computed on the full sample.

In [5]:
def jump_filter(z, n_sigma=3):
    mu, s = z.mean(), z.std()
    hi = mu + n_sigma*s;  lo = mu - n_sigma*s
    return (z > hi) | (z < lo), float(hi), float(lo)

jm_gb, hi_gb, lo_gb = jump_filter(z_gb)
jm_fr, hi_fr, lo_fr = jump_filter(z_fr)
jm_any   = jm_gb | jm_fr
clean_idx = jm_any[~jm_any].index

def ar1(z):
    v = z.loc[clean_idx].values
    x, y = v[:-1], v[1:]
    phi = (x @ y) / (x @ x)
    eps = y - phi * x
    return float(phi), float(eps.std()), eps

phi_gb, sig_gb, eps_gb = ar1(z_gb)
phi_fr, sig_fr, eps_fr = ar1(z_fr)
rho  = float(np.corrcoef(eps_gb, eps_fr)[0, 1])
chol = np.linalg.cholesky([[1.0, rho], [rho, 1.0]])

hl_gb       = np.log(0.5) / np.log(phi_gb)
hl_fr       = np.log(0.5) / np.log(phi_fr)
spread_sig  = np.sqrt(sig_gb**2 + sig_fr**2 - 2*rho*sig_gb*sig_fr)

print(f'Jump dates: GB={jm_gb.sum()}  FR={jm_fr.sum()}  union={jm_any.sum()} '
      f'({100*jm_any.mean():.1f}% of sample)')
print()
print('OU parameters (AR(1), full sample, jump-cleaned, 2016/17-real GBP/MWh):')
print(f'  GB:  phi={phi_gb:.4f}  sigma_d={sig_gb:.2f}  half-life={hl_gb:.1f}d')
print(f'  FR:  phi={phi_fr:.4f}  sigma_d={sig_fr:.2f}  half-life={hl_fr:.1f}d')
print(f'  Residual correlation rho = {rho:.4f}')
print(f'  Implied spread sigma_d   = {spread_sig:.2f} GBP/MWh')

Jump dates: GB=37  FR=37  union=51 (3.1% of sample)

OU parameters (AR(1), full sample, jump-cleaned, 2016/17-real GBP/MWh):
  GB:  phi=0.9209  sigma_d=21.40  half-life=8.4d
  FR:  phi=0.9478  sigma_d=20.09  half-life=12.9d
  Residual correlation rho = 0.5801
  Implied spread sigma_d   = 19.04 GBP/MWh


---
## Section 5: Jump Parameters

Poisson arrival rates and exponential mean jump sizes estimated from the full sample.

In [6]:
def jump_params(z, jm, hi, lo):
    pos = z[jm & (z > hi)] - hi
    neg = lo - z[jm & (z < lo)]
    return {
        'lam_pos': len(pos)/n_days, 'beta_pos': float(pos.mean()) if len(pos) else 1e-9,
        'lam_neg': len(neg)/n_days, 'beta_neg': float(neg.mean()) if len(neg) else 1e-9,
        'n_pos': len(pos), 'n_neg': len(neg),
    }

jp_gb = jump_params(z_gb, jm_gb, hi_gb, lo_gb)
jp_fr = jump_params(z_fr, jm_fr, hi_fr, lo_fr)

print('Jump parameters (full sample, 2016/17-real GBP/MWh):')
print(f'  GB pos: lam={jp_gb["lam_pos"]:.4f}/d  ({jp_gb["n_pos"]} ev)  beta={jp_gb["beta_pos"]:.2f}')
print(f'  GB neg: lam={jp_gb["lam_neg"]:.4f}/d  ({jp_gb["n_neg"]} ev)  beta={jp_gb["beta_neg"]:.2f}')
print(f'  FR pos: lam={jp_fr["lam_pos"]:.4f}/d  ({jp_fr["n_pos"]} ev)  beta={jp_fr["beta_pos"]:.2f}')
print(f'  FR neg: lam={jp_fr["lam_neg"]:.4f}/d  ({jp_fr["n_neg"]} ev)  beta={jp_fr["beta_neg"]:.2f}')

Jump parameters (full sample, 2016/17-real GBP/MWh):
  GB pos: lam=0.0221/d  (37 ev)  beta=74.96
  GB neg: lam=0.0000/d  (0 ev)  beta=0.00
  FR pos: lam=0.0221/d  (37 ev)  beta=50.72
  FR neg: lam=0.0000/d  (0 ev)  beta=0.00


---
## Section 6: Diurnal Spread Shape

Mean within-day spread deviation from the daily average, computed from the full hourly panel. Used to distribute each simulated daily spread across 24 hours for revenue computation.

In [7]:
panel_full = panel.dropna(subset=['spread_gbp'])
daily_s    = panel_full['spread_gbp'].resample('D').mean()
daily_bc   = daily_s.reindex(panel_full.index.normalize()).values
hourly_dev = panel_full['spread_gbp'].values - daily_bc
spread_shape = (pd.Series(hourly_dev, index=panel_full.index)
                .groupby(panel_full.index.hour).mean()
                .reindex(range(24)).fillna(0.0).values)

print('Diurnal spread shape (mean hourly deviation, full sample):')
print(f'  Min={spread_shape.min():.2f}  Max={spread_shape.max():.2f}  '
      f'Peak H{spread_shape.argmax():02d}  Trough H{spread_shape.argmin():02d}')
print(f'  Shape std: {spread_shape.std():.2f} GBP/MWh')

proj_dates  = pd.date_range(f'{PROJ_START_YEAR}-01-01', periods=N_PROJ_YEARS*365, freq='D')
f_gb_proj   = fn_gb(proj_dates)
f_fr_proj   = fn_fr(proj_dates)
proj_years  = proj_dates.year.values
n_proj_days = len(proj_dates)
print(f'\nProjection: {proj_dates[0].date()} → {proj_dates[-1].date()} ({n_proj_days} days)')

Diurnal spread shape (mean hourly deviation, full sample):
  Min=-14.27  Max=14.02  Peak H18  Trough H06
  Shape std: 7.63 GBP/MWh

Projection: 2026-01-01 → 2045-12-26 (7300 days)


---
## Section 7: Monte Carlo Simulation

Bivariate correlated OU with independent Poisson jumps. Revenue = Σ_h |S(d,h)| × CAPACITY × AVAILABILITY × CR / 1e6 (GBPm) per day.

Prices in 2016/17-RPI-real throughout — theta is the full-sample real mean, so revenues are directly comparable to the M2 cap/floor thresholds.

In [8]:
rng        = np.random.default_rng(RANDOM_SEED)
annual_rev = np.zeros((N_PATHS, N_PROJ_YEARS))

for b0 in range(0, N_PATHS, BATCH):
    b1 = min(b0 + BATCH, N_PATHS);  n = b1 - b0
    X_gb = np.zeros(n)
    X_fr = np.zeros(n)
    batch_rev = np.zeros((N_PROJ_YEARS, n))

    for d in range(n_proj_days):
        z     = chol @ rng.standard_normal((2, n))
        X_gb  = phi_gb * X_gb + sig_gb * z[0]
        X_fr  = phi_fr * X_fr + sig_fr * z[1]

        u = rng.random((4, n))
        X_gb += (u[0] < jp_gb['lam_pos']) * rng.exponential(jp_gb['beta_pos'], n)
        X_gb -= (u[1] < jp_gb['lam_neg']) * rng.exponential(jp_gb['beta_neg'], n)
        X_fr += (u[2] < jp_fr['lam_pos']) * rng.exponential(jp_fr['beta_pos'], n)
        X_fr -= (u[3] < jp_fr['lam_neg']) * rng.exponential(jp_fr['beta_neg'], n)

        P_gb_d   = theta_gb + f_gb_proj[d] + X_gb
        P_fr_d   = theta_fr + f_fr_proj[d] + X_fr
        spread_d = P_gb_d - P_fr_d

        S_h   = spread_shape[np.newaxis, :] + spread_d[:, np.newaxis]
        rev_d = np.abs(S_h).sum(axis=1) * CAPACITY_MW * AVAILABILITY * CAPTURE_RATIO / 1e6

        yr_idx = proj_years[d] - PROJ_START_YEAR
        if 0 <= yr_idx < N_PROJ_YEARS:
            batch_rev[yr_idx] += rev_d

    annual_rev[b0:b1] = batch_rev.T
    if b0 % 2000 == 0:
        print(f'  Batch {b0}/{N_PATHS}')

p10 = np.percentile(annual_rev, 10, axis=0)
p50 = np.percentile(annual_rev, 50, axis=0)
p90 = np.percentile(annual_rev, 90, axis=0)
print(f'\nMC complete.')
print(f'  Yr1  P10={p10[0]:.1f}  P50={p50[0]:.1f}  P90={p90[0]:.1f} GBPm')
print(f'  Yr10 P10={p10[9]:.1f}  P50={p50[9]:.1f}  P90={p90[9]:.1f} GBPm')
print(f'  Yr20 P10={p10[19]:.1f}  P50={p50[19]:.1f}  P90={p90[19]:.1f} GBPm')

  Batch 0/10000


  Batch 2000/10000


  Batch 4000/10000


  Batch 6000/10000


  Batch 8000/10000



MC complete.
  Yr1  P10=102.9  P50=129.7  P90=163.7 GBPm
  Yr10 P10=103.6  P50=130.6  P90=166.3 GBPm
  Yr20 P10=102.1  P50=128.6  P90=163.5 GBPm


---
## Section 8: Revenue Results

Annual P10/P50/P90 revenues (2016/17-RPI-real GBPm). Because theta is the full-sample mean (no trend), revenues are stationary around the historically-estimated long-run level.

In [9]:
print(f'  {"Year":>6}  {"P10":>8}  {"P50":>8}  {"P90":>8}  {"P90-P10":>10}')
print('  ' + '-'*48)
for i in range(N_PROJ_YEARS):
    yr   = PROJ_START_YEAR + i
    note = '  <- yr10' if i==9 else ('  <- yr20' if i==19 else '')
    print(f'  {yr:>6}  {p10[i]:>8.1f}  {p50[i]:>8.1f}  {p90[i]:>8.1f}  '
          f'{p90[i]-p10[i]:>10.1f}{note}')
print()
print(f'  Mean P50: {p50.mean():.1f} GBPm   P10: {p10.mean():.1f}   P90: {p90.mean():.1f}')

    Year       P10       P50       P90     P90-P10
  ------------------------------------------------
    2026     102.9     129.7     163.7        60.8
    2027     103.8     130.4     165.0        61.2
    2028     104.4     131.0     165.9        61.5
    2029     104.2     130.4     165.2        60.9
    2030     103.8     130.8     166.9        63.1
    2031     104.2     131.4     166.0        61.8
    2032     104.0     130.6     166.4        62.4
    2033     103.4     130.7     165.9        62.6
    2034     103.7     131.3     165.6        61.9
    2035     103.6     130.6     166.3        62.6  <- yr10
    2036     104.2     131.4     166.6        62.4
    2037     103.7     130.8     166.0        62.3
    2038     103.9     130.8     165.9        62.0
    2039     103.8     131.2     166.5        62.7
    2040     104.5     131.8     167.2        62.7
    2041     104.0     131.1     165.7        61.7
    2042     104.1     131.3     166.6        62.5
    2043     103.2    

---
## Section 9: Physical Capacity Ceiling Check

In [10]:
max_spread  = daily['spread_gbp'].abs().max()
phys_ceil   = 24 * max_spread * CAPACITY_MW * AVAILABILITY * CAPTURE_RATIO * 365 / 1e6
n_breaches  = int((annual_rev > phys_ceil).sum())

print(f'Physical ceiling: {phys_ceil:.1f} GBPm/yr  (max |spread|={max_spread:.2f} GBP/MWh)')
if n_breaches > 0:
    yrs = [PROJ_START_YEAR+j for j in range(N_PROJ_YEARS) if (annual_rev[:,j]>phys_ceil).any()]
    print(f'WARNING: {n_breaches} path-year cells exceed ceiling  |  years: {yrs}')
    print(f'Max simulated: {annual_rev.max():.1f} GBPm')
else:
    print('OK — no path-year exceeds the physical ceiling.')

Physical ceiling: 478.5 GBPm/yr  (max |spread|=218.35 GBP/MWh)
OK — no path-year exceeds the physical ceiling.


---
## Section 10: Revenue Fan Chart

In [11]:
proj_yrs = list(range(PROJ_START_YEAR, PROJ_START_YEAR + N_PROJ_YEARS))

fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(proj_yrs, p10, p90, alpha=0.2, color='steelblue', label='P10–P90 band')
ax.plot(proj_yrs, p50, 'b-o', ms=4, lw=1.5, label='P50')
ax.plot(proj_yrs, p10, 'b--', lw=0.7, alpha=0.5, label='P10')
ax.plot(proj_yrs, p90, 'b--', lw=0.7, alpha=0.5, label='P90')
ax.axhline(37.8, color='red',   lw=1.2, ls=':', label='W1 floor £37.8m (2016/17-real)')
ax.axhline(61.6, color='green', lw=1.2, ls=':', label='W1 cap £61.6m (2016/17-real)')
ax.set_xlabel('Year')
ax.set_ylabel('Annual Revenue (2016/17-real GBPm)')
ax.set_title(
    'Fig M1-1. IFA2 Bivariate OU — Full Sample, No Calibration (10,000 paths)\n'
    f'theta_spread={theta_gb-theta_fr:.2f} GBP/MWh  |  '
    f'phi_GB={phi_gb:.3f}  phi_FR={phi_fr:.3f}  |  rho={rho:.3f}',
    fontweight='bold')
ax.legend(fontsize=9, ncol=3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'figM1_revenue_fan.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: figM1_revenue_fan.png')

Saved: figM1_revenue_fan.png


---
## Section 11: Window 1 Cap/Floor Breach Probabilities

IFA2 is a **Window 1** project. The W1 cap and floor are **RPI-indexed**, so their real values are constant in 2016/17-RPI-real terms — directly comparable to `annual_rev`.

| Threshold | 2016/17-real | Nominal indexation |
|-----------|-------------|--------------------|
| W1 Cap    | £61.6m      | RPI + 3.24%/yr     |
| W1 Floor  | £37.8m      | RPI + 3.24%/yr     |

Assessment periods aligned to the regulatory structure (P1=2021–25, P2=2026–30, …):

| Period | Years     | Note |
|--------|-----------|------|
| P1     | 2021–2025 | Pre-projection (historical actuals) |
| P2     | 2026–2030 | First full MC period |
| P3     | 2031–2035 | |
| P4     | 2036–2040 | |
| P5     | 2041–2045 | 2045 = final year of regime, avoids period-stub artifact |

Discount rate = 3.945% real ODR. All periods are 5 full years.

In [12]:
W1_CAP_REAL   = 61.6    # £61.6m (2016/17-RPI-real)
W1_FLOOR_REAL = 37.8    # £37.8m (2016/17-RPI-real)
ODR           = 0.03945

YRS = list(range(PROJ_START_YEAR, PROJ_START_YEAR + N_PROJ_YEARS))

print('W1 Annual Breach Probabilities (2016/17-RPI-real)')
print(f'  Cap = £{W1_CAP_REAL}m  |  Floor = £{W1_FLOOR_REAL}m  (constant in 2016/17-real)')
print()
print(f'  {"Year":>6}  {"P10":>8}  {"P50":>8}  {"P90":>8}  {"P>Cap":>8}  {"P<Floor":>8}')
print('  ' + '-'*60)

show_yrs = {2026, 2028, 2030, 2032, 2035, 2040, 2044}
for i, yr in enumerate(YRS):
    if yr in show_yrs:
        rv = annual_rev[:, i]
        p10_v, p50_v, p90_v = np.percentile(rv, [10, 50, 90])
        p_cap   = 100 * (rv > W1_CAP_REAL).mean()
        p_floor = 100 * (rv < W1_FLOOR_REAL).mean()
        print(f'  {yr:>6}  {p10_v:>8.1f}  {p50_v:>8.1f}  {p90_v:>8.1f}  {p_cap:>7.1f}%  {p_floor:>7.1f}%')

print()
W1_PERIODS = [
    ('P2', list(range(2026, 2031))),
    ('P3', list(range(2031, 2036))),
    ('P4', list(range(2036, 2041))),
    ('P5', list(range(2041, 2046))),
]

print('W1 NPV Assessment Periods (2016/17-RPI-real, ODR=3.945%)')
print(f'  {"Period":>8}  {"Years":>12}  {"P50 NPV":>10}  {"Cap NPV":>10}  {"Flr NPV":>10}  {"P>Cap":>8}  {"P<Flr":>8}')
print('  ' + '-'*84)

w1_npv = {}
for pname, yr_rng in W1_PERIODS:
    yrs_in = [yr for yr in yr_rng if yr in YRS]
    if not yrs_in:
        continue
    disc     = np.array([1.0 / (1 + ODR)**k for k in range(len(yrs_in))])
    cols     = [YRS.index(yr) for yr in yrs_in]
    rev_mat  = annual_rev[:, cols]
    npv_vec  = (rev_mat * disc[np.newaxis, :]).sum(axis=1)
    ann      = disc.sum()
    npv_cap  = W1_CAP_REAL * ann
    npv_flr  = W1_FLOOR_REAL * ann
    p50_npv  = float(np.percentile(npv_vec, 50))
    p_cap    = 100 * (npv_vec > npv_cap).mean()
    p_floor  = 100 * (npv_vec < npv_flr).mean()
    yr_lbl   = f'{yrs_in[0]}\u2013{yrs_in[-1]}'
    w1_npv[pname] = {'p50':p50_npv, 'npv_cap':npv_cap, 'npv_flr':npv_flr,
                     'p_cap':p_cap, 'p_flr':p_floor, 'yrs':yr_lbl}
    print(f'  {pname:>8}  {yr_lbl:>12}  {p50_npv:>10.1f}  {npv_cap:>10.1f}  {npv_flr:>10.1f}  {p_cap:>7.1f}%  {p_floor:>7.1f}%')

W1 Annual Breach Probabilities (2016/17-RPI-real)
  Cap = £61.6m  |  Floor = £37.8m  (constant in 2016/17-real)

    Year       P10       P50       P90     P>Cap   P<Floor
  ------------------------------------------------------------
    2026     102.9     129.7     163.7    100.0%      0.0%
    2028     104.4     131.0     165.9    100.0%      0.0%
    2030     103.8     130.8     166.9    100.0%      0.0%
    2032     104.0     130.6     166.4    100.0%      0.0%
    2035     103.6     130.6     166.3    100.0%      0.0%
    2040     104.5     131.8     167.2    100.0%      0.0%
    2044     103.5     131.0     166.4    100.0%      0.0%

W1 NPV Assessment Periods (2016/17-RPI-real, ODR=3.945%)
    Period         Years     P50 NPV     Cap NPV     Flr NPV     P>Cap     P<Flr
  ------------------------------------------------------------------------------------
        P2     2026–2030       613.6       285.5       175.2    100.0%      0.0%
        P3     2031–2035       615.8       28

---
## Section 12: Window 3 Cap/Floor Breach Probabilities

W3 thresholds are defined in **2024-CPIH-real** terms. Revenue conversion: `rev_2024cpih = rev_1617real × (RPI_t / RPI_1617) / CPIH_t`

where `CPIH_t = 1.025^(year − 2024)` (OBR 2.5%/yr from ONS base 132.9 in 2024).

| Threshold | 2024-CPIH-real | Components |
|-----------|---------------|------------|
| W3 Cap    | £79.885m      | PCL £67.05m + PCAC £12.83m |
| W3 Floor  | £49.956m      | PFL £36.47m + PCAF £13.49m |

Scale factors RPIt/CPIHt: 2026≈1.311, 2030≈1.349, 2035≈1.399, 2040≈1.450, 2045≈1.503

Assessment periods: P2=2026–2030, P3=2031–2035, P4=2036–2040, P5=2041–2045 (5 full years each).

In [13]:
W3_CAP_CPIH   = 79.885   # £79.885m (2024-CPIH-real)
W3_FLOOR_CPIH = 49.956   # £49.956m (2024-CPIH-real)

def _rpit(yr):
    return _ALL_RPI.get(yr, _ALL_RPI[max(_ALL_RPI)]) / _RPI_1617

def _cpiht(yr):
    return 1.025 ** (yr - 2024)

def to_cpih(rev_1617r, yr):
    return rev_1617r * _rpit(yr) / _cpiht(yr)

print('W3 Annual Breach Probabilities (2024-CPIH-real)')
print(f'  Cap = £{W3_CAP_CPIH}m  |  Floor = £{W3_FLOOR_CPIH}m')
print()
print(f'  {"Year":>6}  {"Scale":>7}  {"P10":>8}  {"P50":>8}  {"P90":>8}  {"P>Cap":>8}  {"P<Floor":>8}')
print('  ' + '-'*68)

for i, yr in enumerate(YRS):
    if yr in show_yrs:
        sc       = _rpit(yr) / _cpiht(yr)
        rv_cpih  = to_cpih(annual_rev[:, i], yr)
        p10_v, p50_v, p90_v = np.percentile(rv_cpih, [10, 50, 90])
        p_cap    = 100 * (rv_cpih > W3_CAP_CPIH).mean()
        p_floor  = 100 * (rv_cpih < W3_FLOOR_CPIH).mean()
        print(f'  {yr:>6}  {sc:>7.4f}  {p10_v:>8.1f}  {p50_v:>8.1f}  {p90_v:>8.1f}  {p_cap:>7.1f}%  {p_floor:>7.1f}%')

print()
W3_PERIODS = [
    ('P2', list(range(2026, 2031))),
    ('P3', list(range(2031, 2036))),
    ('P4', list(range(2036, 2041))),
    ('P5', list(range(2041, 2046))),
]

print('W3 NPV Assessment Periods (2024-CPIH-real, ODR=3.945%)')
print(f'  {"Period":>8}  {"Years":>12}  {"P50 NPV":>10}  {"W3 Cap":>10}  {"W3 Flr":>10}  {"P>Cap":>8}  {"P<Flr":>8}')
print('  ' + '-'*84)

w3_npv = {}
for pname, yr_rng in W3_PERIODS:
    yrs_in   = [yr for yr in yr_rng if yr in YRS]
    if not yrs_in:
        continue
    disc     = np.array([1.0 / (1 + ODR)**k for k in range(len(yrs_in))])
    cols     = [YRS.index(yr) for yr in yrs_in]
    rev_cpih = np.column_stack([to_cpih(annual_rev[:, c], YRS[c]) for c in cols])
    npv_vec  = (rev_cpih * disc[np.newaxis, :]).sum(axis=1)
    ann      = disc.sum()
    npv_cap  = W3_CAP_CPIH * ann
    npv_flr  = W3_FLOOR_CPIH * ann
    p50_npv  = float(np.percentile(npv_vec, 50))
    p_cap    = 100 * (npv_vec > npv_cap).mean()
    p_floor  = 100 * (npv_vec < npv_flr).mean()
    yr_lbl   = f'{yrs_in[0]}\u2013{yrs_in[-1]}'
    w3_npv[pname] = {'p50':p50_npv, 'npv_cap':npv_cap, 'npv_flr':npv_flr,
                     'p_cap':p_cap, 'p_flr':p_floor, 'yrs':yr_lbl}
    print(f'  {pname:>8}  {yr_lbl:>12}  {p50_npv:>10.1f}  {npv_cap:>10.1f}  {npv_flr:>10.1f}  {p_cap:>7.1f}%  {p_floor:>7.1f}%')

W3 Annual Breach Probabilities (2024-CPIH-real)
  Cap = £79.885m  |  Floor = £49.956m

    Year    Scale       P10       P50       P90     P>Cap   P<Floor
  --------------------------------------------------------------------
    2026   1.3093     134.7     169.8     214.4    100.0%      0.0%
    2028   1.3283     138.6     174.0     220.4    100.0%      0.0%
    2030   1.3475     139.9     176.2     224.9    100.0%      0.0%
    2032   1.3670     142.1     178.5     227.4    100.0%      0.0%
    2035   1.3968     144.8     182.4     232.3    100.0%      0.0%
    2040   1.4480     151.3     190.8     242.0    100.0%      0.0%
    2044   1.4903     154.3     195.3     248.0    100.0%      0.0%

W3 NPV Assessment Periods (2024-CPIH-real, ODR=3.945%)
    Period         Years     P50 NPV      W3 Cap      W3 Flr     P>Cap     P<Flr
  ------------------------------------------------------------------------------------
        P2     2026–2030       814.7       370.2       231.5    100.0%    

---
## Section 13: W1 vs W3 Summary

Side-by-side regulatory comparison. **Key structural difference:**

- W1 cap grows with RPI → rises faster than revenues in real terms → cap bite eases over time
- W3 cap is fixed in CPIH-real → cap clawback risk depends on whether spread mean reverts below W3 cap level

> *Note: floor breach is 0% in both regimes. The full-sample theta (crisis-era mean) keeps revenues far above floor throughout.*

In [14]:
print('W1 vs W3 NPV Assessment Summary')
print()
hdr = f'{"Period":>6}  {"Years":>12}  {"P50 NPV":>10}'
hdr += f'  {"W1Cap":>9}  {"W1 P>Cap":>10}  {"W3Cap":>9}  {"W3 P>Cap":>10}'
print('  ' + hdr)
print('  ' + '-'*90)

for p in ['P2','P3','P4','P5']:
    if p not in w1_npv or p not in w3_npv:
        continue
    w1, w3 = w1_npv[p], w3_npv[p]
    line  = f'{p:>6}  {w1["yrs"]:>12}  {w1["p50"]:>10.1f}'
    line += f'  {w1["npv_cap"]:>9.1f}  {w1["p_cap"]:>9.1f}%  {w3["npv_cap"]:>9.1f}  {w3["p_cap"]:>9.1f}%'
    print('  ' + line)

print()
print('Floor breach (P<Flr): 0% in all periods for both W1 and W3.')
print()
print('Interpretation note:')
print('  Revenues modelled on full 2021-26 sample (crisis-era theta ~£23 GBP/MWh spread).')
print('  P50 revenues (~£130m/yr in 2016/17-real) sit far above both regulatory caps.')
print('  Under W1: cap clawback is near-certain in all periods at these revenue levels.')
print('  Under W3: same conclusion — W3 cap lower than W1 cap in 2016/17-real equivalent.')
print('  Floor protection is not load-bearing under full-sample parameters.')

W1 vs W3 NPV Assessment Summary

  Period         Years     P50 NPV      W1Cap    W1 P>Cap      W3Cap    W3 P>Cap
  ------------------------------------------------------------------------------------------
      P2     2026–2030       613.6      285.5      100.0%      370.2      100.0%
      P3     2031–2035       615.8      285.5      100.0%      370.2      100.0%
      P4     2036–2040       617.4      285.5      100.0%      370.2      100.0%
      P5     2041–2045       613.6      285.5      100.0%      370.2      100.0%

Floor breach (P<Flr): 0% in all periods for both W1 and W3.

Interpretation note:
  Revenues modelled on full 2021-26 sample (crisis-era theta ~£23 GBP/MWh spread).
  P50 revenues (~£130m/yr in 2016/17-real) sit far above both regulatory caps.
  Under W1: cap clawback is near-certain in all periods at these revenue levels.
  Under W3: same conclusion — W3 cap lower than W1 cap in 2016/17-real equivalent.
  Floor protection is not load-bearing under full-sample par